**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Chapter 15: A Short Introduction to Loops

A **loop** tells Python to repeat the same block of code once for every item in a collection. Loops are useful whenever you would otherwise copy and paste the same calculation several times with only a name or number changed. In `Ex15-TestingCAPM.ipynb` we estimate a CAPM regression for each of the ten beta-sorted decile portfolios. Rather than writing ten nearly identical regressions, we write the regression once and let a loop run it ten times, storing each result as we go. This notebook introduces `for` loops in general and then walks through that loop line by line. The examples use the same small, made-up set of monthly portfolio returns as `Ex06-pandas_intro.ipynb`, created directly in the notebook, so it runs without downloading or reading any files.

## Learning objectives

By the end of this notebook, you should be able to:

- Write a `for` loop and explain the role of the loop variable, the colon, and indentation.
- Use `range()` and f-strings to build a list of names such as `dec1` through `dec10`.
- Store results from each pass through a loop in a list or dictionary instead of overwriting them.
- Loop over the columns of a DataFrame and combine the stored results into a new DataFrame.
- Explain what each line of the CAPM loop in `Ex15-TestingCAPM.ipynb` does and why it works.

## What is a loop?

The basic form of a `for` loop is:

```python
for item in collection:
    # code that runs once for each item
```

- `collection` is anything Python can step through one element at a time, such as a list, a `range`, or the columns of a DataFrame.
- `item` is the **loop variable**. On each pass, or **iteration**, Python assigns the next element of the collection to it. You can choose any name.
- The line ending in a colon `:` starts the loop.
- The **indented** lines below it form the **body** of the loop. Python runs the body once per element. The first line that is not indented runs after the loop has finished.

In [5]:
portfolios = ['Low-momentum', 'High-momentum', 'Market']

for name in portfolios:
    # This indented line runs once for each element of the list.
    print('Now working on:', name)

# This line is not indented, so it runs once, after the loop finishes.
print('Done. The last value of name is:', name)

Now working on: Low-momentum
Now working on: High-momentum
Now working on: Market
Done. The last value of name is: Market


## Loop over numbers with `range()`

`range(start, stop)` produces the whole numbers from `start` up to, **but not including**, `stop`. So `range(1, 11)` produces 1, 2, ..., 10. Combined with an f-string, which inserts the value inside `{}` into the text, a loop can build a list of column names. We start with an empty list `[]` and use `.append()` to add one name on each iteration.

In [6]:
print('range(1, 11) contains:', list(range(1, 11)))

# Build the decile column names one at a time.
decile_names = []
for i in range(1, 11):
    decile_names.append(f'dec{i}')

print(decile_names)

range(1, 11) contains: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
['dec1', 'dec2', 'dec3', 'dec4', 'dec5', 'dec6', 'dec7', 'dec8', 'dec9', 'dec10']


### A shortcut: list comprehensions

Building a list with a loop is so common that Python has a one-line shortcut called a **list comprehension**. The code below produces exactly the same list as the loop above. This is the line that `Ex15-TestingCAPM.ipynb` uses to define `portfolio_columns`. You saw the same idea in `Ex06-Ken French_Portfolio_Returns.ipynb` and `Ex10-efficient_frontier.ipynb`.

In [7]:
portfolio_columns = [f'dec{i}' for i in range(1, 11)]

print(portfolio_columns)
print('Same as the loop version?', portfolio_columns == decile_names)

['dec1', 'dec2', 'dec3', 'dec4', 'dec5', 'dec6', 'dec7', 'dec8', 'dec9', 'dec10']
Same as the loop version? True


## Store results from each iteration

A loop variable, or any variable assigned inside the loop, is **overwritten** on every iteration. If we only assign a result to a variable, we keep the result from the last iteration and lose all the others. To keep every result, create an empty container **before** the loop and add one result to it **inside** the loop. This is called the **accumulator pattern**.

Here we create the sample returns and compute the average excess return of each portfolio, first the wrong way and then the right way.

In [8]:
import pandas as pd

# Create six months of made-up portfolio returns indexed by month-end date.
portfolio_returns = pd.DataFrame(
    {
        'Low-momentum': [0.020, -0.015, 0.010, 0.005, -0.010, 0.018],
        'High-momentum': [0.028, -0.005, 0.016, 0.012, -0.004, 0.025],
        'Market': [0.022, -0.010, 0.012, 0.008, -0.006, 0.020],
        'Risk-free': [0.004, 0.004, 0.004, 0.004, 0.004, 0.004],
    },
    index=pd.date_range('2024-01-31', periods=6, freq='ME', name='Date'),
)

portfolio_returns

,Low-momentum,High-momentum,Market,Risk-free
Date,,,,
2024-01-31,0.020,0.028,0.022,0.004
2024-02-29,-0.015,-0.005,-0.010,0.004
2024-03-31,0.010,0.016,0.012,0.004
2024-04-30,0.005,0.012,0.008,0.004
2024-05-31,-0.010,-0.004,-0.006,0.004
2024-06-30,0.018,0.025,0.020,0.004


In [9]:
# Wrong: avg_excess is overwritten on every iteration.
for column in ['Low-momentum', 'High-momentum', 'Market']:
    avg_excess = (portfolio_returns[column] - portfolio_returns['Risk-free']).mean()

print('Only the last result survives:', float(avg_excess))

# Right: create a container before the loop and add to it inside the loop.
avg_excess_list = []
for column in ['Low-momentum', 'High-momentum', 'Market']:
    avg_excess = (portfolio_returns[column] - portfolio_returns['Risk-free']).mean()
    avg_excess_list.append(float(avg_excess))

print('Every result is kept:', avg_excess_list)

Only the last result survives: 0.0036666666666666666
Every result is kept: [0.0006666666666666666, 0.008, 0.0036666666666666666]


### Keep the labels with a dictionary or a Series

A plain list stores the numbers but forgets which portfolio each number belongs to. A **dictionary** stores each result under a key, here the column name. A dictionary converts directly to a Pandas Series whose index is the keys.

In [10]:
avg_excess_dict = {}
for column in ['Low-momentum', 'High-momentum', 'Market']:
    avg_excess_dict[column] = float((portfolio_returns[column] - portfolio_returns['Risk-free']).mean())

print(avg_excess_dict)

pd.Series(avg_excess_dict, name='Average excess return')

{'Low-momentum': 0.0006666666666666666, 'High-momentum': 0.008, 'Market': 0.0036666666666666666}


Low-momentum     0.000667
High-momentum    0.008000
Market           0.003667
Name: Average excess return, dtype: float64

### Store several numbers per iteration

Sometimes each iteration produces more than one number. In the CAPM notebook each regression produces an estimate, a lower confidence bound, and an upper confidence bound. A convenient approach is to store one **Series per iteration** in a list. Two Pandas tools make this work:

- `series.rename('label')` gives the Series a name.
- `pd.DataFrame(list_of_series)` stacks the Series as rows. Each Series' name becomes a row label in the index, and the Series' own index labels become the columns.

In [11]:
rows = []
for column in ['Low-momentum', 'High-momentum', 'Market']:
    excess = portfolio_returns[column] - portfolio_returns['Risk-free']
    stats = pd.Series({'mean': excess.mean(), 'min': excess.min(), 'max': excess.max()})
    rows.append(stats.rename(column))

# Look at one stored Series: its name is the portfolio, its index holds the statistics.
print(rows[0])

# Stack the stored Series into a DataFrame with one row per portfolio.
summary = pd.DataFrame(rows)
summary.index.name = 'portfolio'
summary

mean    0.000667
min    -0.019000
max     0.016000
Name: Low-momentum, dtype: float64


,mean,min,max
portfolio,,,
Low-momentum,0.000667,-0.019,0.016
High-momentum,0.008000,-0.009,0.024
Market,0.003667,-0.014,0.018


## The loop in `Ex15-TestingCAPM.ipynb`

We now have every piece needed to read the CAPM loop. Here is the code from `Ex15-TestingCAPM.ipynb`:

```python
portfolio_columns = [f'dec{i}' for i in range(1, 11)]

rows = []
for column in portfolio_columns:
    regression = fm.linear_regression_summary(
        X=df['mkt-rf'],
        Y=df[column] - df['rf'],
    )
    rows.append(regression.loc['const'].rename(column))

alphas = pd.DataFrame(rows)
alphas.index.name = 'decile'
```

Line by line:

1. `portfolio_columns = [...]` builds the list `['dec1', ..., 'dec10']`, the names of the ten portfolio columns in `df`.
2. `rows = []` creates an empty list **before** the loop to store one result per decile.
3. `for column in portfolio_columns:` sets `column` to `'dec1'` on the first iteration, `'dec2'` on the second, and so on through `'dec10'`.
4. `df[column] - df['rf']` selects the current decile's returns and subtracts the risk-free rate to get its excess return. Because `column` changes on each iteration, the same line of code selects a different portfolio each time.
5. `fm.linear_regression_summary(...)` regresses that excess return on the market excess return `df['mkt-rf']`. It returns a small DataFrame with rows `const` and `mkt-rf` and columns `coef`, `ci_lower`, and `ci_upper`.
6. `regression.loc['const']` keeps only the intercept row, which is the CAPM alpha and its confidence interval, as a Series.
7. `.rename(column)` labels that Series with the decile name so we know which portfolio it came from.
8. `rows.append(...)` adds it to the list. Without this line, `regression` would be overwritten on the next iteration and the result would be lost.
9. After the loop, `pd.DataFrame(rows)` stacks the ten stored Series into one table with one row per decile.

### Why it works

The ten regressions differ only in **which column** is used as the portfolio return. Everything else, the market factor, the risk-free rate, and the regression function, stays the same. A loop is exactly the tool for repeating a calculation where only one name changes. Storing a labeled result on each pass, and combining the results once the loop finishes, turns ten separate regressions into one table that is easy to print and plot.

### Try the same loop on the sample data

The cell below runs the same loop on the small sample data set. The only changes are the column names: our portfolios are `Low-momentum` and `High-momentum` instead of `dec1` through `dec10`, and we compute the market excess return from the `Market` and `Risk-free` columns. First we run one regression by hand to see what a single iteration produces. With only six months of made-up data the estimates are not meaningful; the point is the structure of the code.

In [12]:
# Import packages
%pip install -q --upgrade farms

import farms as fm

# Market excess return, which plays the role of df['mkt-rf'].
market_excess = (portfolio_returns['Market'] - portfolio_returns['Risk-free']).rename('mkt-rf')

# One iteration by hand: the full regression table for a single portfolio.
regression = fm.linear_regression_summary(
    X=market_excess,
    Y=portfolio_returns['Low-momentum'] - portfolio_returns['Risk-free'],
)
print('Full regression table:')
print(regression)
print('\nThe intercept row we keep:')
print(regression.loc['const'])

Note: you may need to restart the kernel to use updated packages.
Full regression table:
            coef  ci_lower  ci_upper
const  -0.003335 -0.003827 -0.002844
mkt-rf  1.091394  1.052460  1.130327

The intercept row we keep:
coef       -0.003335
ci_lower   -0.003827
ci_upper   -0.002844
Name: const, dtype: float64


In [13]:
sample_columns = ['Low-momentum', 'High-momentum']

# Estimate the CAPM regression for each portfolio and keep the intercept row.
rows = []
for column in sample_columns:
    regression = fm.linear_regression_summary(
        X=market_excess,
        Y=portfolio_returns[column] - portfolio_returns['Risk-free'],
    )
    rows.append(regression.loc['const'].rename(column))

# Combine the alphas and confidence intervals into one DataFrame.
alphas = pd.DataFrame(rows)
alphas.index.name = 'portfolio'
alphas

,coef,ci_lower,ci_upper
portfolio,,,
Low-momentum,-0.003335,-0.003827,-0.002844
High-momentum,0.004121,0.002623,0.005620


## Common mistakes

- **Forgetting the colon** at the end of the `for` line causes a `SyntaxError`.
- **Inconsistent indentation** causes an `IndentationError`, or puts a line inside or outside the loop by accident. Jupyter indents with four spaces automatically after a line ending in `:`.
- **Creating the container inside the loop.** If `rows = []` is placed inside the loop, the list is emptied on every iteration and only the last result survives.
- **Forgetting to store the result.** If you compute a value but never `.append()` it or assign it to a dictionary, it is overwritten on the next iteration.
- **Off-by-one ranges.** `range(1, 10)` stops at 9. Use `range(1, 11)` to include 10.

## Key terms

- **Loop:** code that repeats a block of statements once for each item in a collection.
- **Iteration:** one pass through the body of a loop.
- **Loop variable:** the name, such as `column`, that takes the value of the current item on each iteration.
- **Body:** the indented lines that run on every iteration.
- **`range(start, stop)`:** the whole numbers from `start` up to, but not including, `stop`.
- **Accumulator pattern:** creating an empty list or dictionary before a loop and adding one result to it on each iteration.
- **List comprehension:** a one-line way to build a list, such as `[f'dec{i}' for i in range(1, 11)]`.